In [17]:
# Motion detection  - Consecutive Frame difference

import cv2
cap = cv2.VideoCapture(0)
ret, previous = cap.read()
previous_gray = cv2.cvtColor(previous, cv2.COLOR_BGR2GRAY)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    diff = cv2.absdiff(previous_gray,gray)
    _, mask = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)
    cv2.imshow("Original", frame)  
    
    cv2.imshow("Motion detection  - Consecutive Frame", mask)
    if cv2.waitKey(1)&0xFF ==27:
        break
cap.release()
cv2.destroyAllWindows()

In [10]:
# Robust motion detection - blur, threshold, morphology

import cv2
cap = cv2.VideoCapture(0)
ret, previous = cap.read()
previous_gray = cv2.cvtColor(previous, cv2.COLOR_BGR2GRAY)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))


while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21,21),0)
    
    diff = cv2.absdiff(previous_gray,gray)
    _, mask = cv2.threshold(diff, 125, 255, cv2.THRESH_BINARY)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.dilate(mask, kernel, iterations=2)

    cv2.putText(frame, "Motion Detected" if cv2.countNonZero(mask)>1000 else "No Motion!!",
    (20,40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
    cv2.imshow("mask", mask)  
    cv2.imshow("Motion detection", frame)
    if cv2.waitKey(1)&0xFF ==27:
        break
cap.release()
cv2.destroyAllWindows()

In [16]:
# Motion alarm = > event looging and frame capture

# Robust motion detection - blur, threshold, morphology

import cv2
import time
from datetime import datetime

cap = cv2.VideoCapture(0)

# Read first frame
ret, previous = cap.read()

if not ret:
    print("Cannot access camera")
    cap.release()
    exit()

# Convert first frame to grayscale + blur
previous_gray = cv2.cvtColor(previous, cv2.COLOR_BGR2GRAY)
previous_gray = cv2.GaussianBlur(previous_gray, (21, 21), 0)

# Motion event cooldown
countdown = 3
last_event = 0

# Morphology kernel
kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (5, 5)
)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Current frame
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)

    # Compare previous and current frame
    diff = cv2.absdiff(previous_gray, gray)

    # Threshold
    _, mask = cv2.threshold(
        diff,
        30,
        255,
        cv2.THRESH_BINARY
    )

    # Remove small gaps/noise
    mask = cv2.dilate(
        mask,
        kernel,
        iterations=2
    )

    # Calculate motion
    motion_pixels = cv2.countNonZero(mask)

    motion = motion_pixels > 2500

    # Motion event
    if motion and time.time() - last_event > countdown:

        name = datetime.now().strftime(
            "motion_%Y%m%d_%H%M%S.jpg"
        )

        cv2.imwrite(name, frame)

        print("Motion Event:", name)

        last_event = time.time()

    # Display status
    status = "Motion Detected" if motion else "Waiting!!"

    cv2.putText(
        frame,
        status,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    # Display
    cv2.imshow("Motion Alarm", frame)

    # Update previous frame
    previous_gray = gray

    # ESC
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


Motion Event: motion_20260819_112413.jpg
Motion Event: motion_20260819_112423.jpg
Motion Event: motion_20260819_112432.jpg
Motion Event: motion_20260819_112445.jpg
Motion Event: motion_20260819_112448.jpg
